In [14]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()
api = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="meta-llama/Llama-4-Scout-17B-16E", groq_api_key = api, temperature=0)

In [ ]:
msg = llm.invoke(["hello"])

### testing ragas

In [17]:
from datasets import load_dataset, DownloadMode

cuad = load_dataset(
    "theatticusproject/cuad", "qa"
)["train"]

print(f"Loaded {len(cuad)} examples")

ValueError: BuilderConfig 'qa' not found. Available: ['default']

In [18]:
from datasets import get_dataset_config_names

print(get_dataset_config_names("theatticusproject/cuad"))

['default']


In [ ]:
cuad

Dataset({
    features: ['pdf'],
    num_rows: 511
})

In [7]:
answerable = []
for row in cuad:
    # Check if 'answers' exists and has content
    ans_data = row.get("answers")
    if ans_data and len(ans_data.get("text", [])) > 0:
        answerable.append(row)

if not answerable:
    print("No answerable rows found. Check dataset schema!")


No answerable rows found. Check dataset schema!


In [10]:
cuad

Dataset({
    features: ['pdf'],
    num_rows: 511
})

In [8]:
cuad.features.keys()

dict_keys(['pdf'])

In [ ]:
cuad[0]
output: {'pdf': <pdfplumber.pdf.PDF at 0x2c283459c10>}

{'pdf': <pdfplumber.pdf.PDF at 0x2c283459c10>}

In [ ]:
cuad.column_names
output: ['pdf']

['pdf']

In [14]:
dataset = load_dataset("json", data_files="https://huggingface.co/datasets/theatticusproject/cuad/resolve/main/CUADv1.json", field="data")

FileNotFoundError: Unable to find 'hf://datasets/theatticusproject/cuad@main/CUADv1.json'

In [11]:
dataset = load_dataset("theatticusproject/cuad", "cuad")["train"]

ValueError: BuilderConfig 'cuad' not found. Available: ['default']

In [11]:
from datasets import load_dataset
from pipeline import RAGPipeline
import random
import logging

logger = logging.getLogger(__name__)

ModuleNotFoundError: No module named 'pipeline'

In [9]:
def build_golden_dataset(n_samples: int = 35):

    pipeline = RAGPipeline()

    answerable = [row for row in cuad if row["answers"]["text"]]

    sampled = random.sample(answerable, min(n_samples, len(answerable)))

    golden = []

    for i, row in enumerate(sampled):
        logger.info(f"Processing {i+1}/{len(sampled)}: {row['question']}")

        try:
            result = pipeline.run(row["question"])
            answer = pipeline.llm.generate_response(
                result["rewritten_query"], result["chunks"]
            )
            golden.append(
                {
                    "question": row["question"],
                    "ground_truth": row["answers"]["text"][0],
                    "answer": answer["answer"],
                    "context": [c["text"] for c in result["chunks"]]
                }
            )

        except Exception as e:
            logger.error(f"skipped: {e}")
            continue

    return golden


In [10]:
build_golden_dataset()

NameError: name 'RAGPipeline' is not defined

## load clause master csv

In [9]:
import pandas as pd

data = pd.read_csv("../data/master_clauses.csv")
data.head(20)["Parties"].tolist()

["['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', 'Marketing Affiliate', 'MOUNT KNOWLEDGE HOLDINGS INC.', 'Company']",
 "['EuroMedia Holdings Corp.', 'Rogers', 'Rogers Cable Communications Inc.', 'Licensor']",
 "['Producer', 'Fulucai Productions Ltd.', 'ConvergTV', 'CONVERGTV, INC.']",
 "['PSiTech Corporation', 'Licensor', 'Licensee', 'Empirical Ventures, Inc.']",
 "['YOU ON DEMAND HOLDINGS, INC.', 'Licensor', 'Licensee', 'Beijing Sun Seven Stars Culture Development Limited']",
 "['PrimeCall', 'deltathree.com, Inc. (formerly known as Delta Three, Inc.)', 'RSL COM PrimeCall, Inc.', 'DeltaThree']",
 "['Women.com', 'eDiets', 'WOMEN.COM NETWORKS, INC.', 'EDIETS.COM, INC.']",
 "['d/b/a Time Life Music', 'Integrity', 'TL', 'TIME LIFE, INC.', 'INTEGRITY INCORPORATED']",
 '[\'Endorser\', \'collectively, Lender, Endorser, and Fitness are referred to as the "AS Parties"\', \'MusclePharm Corporation\', \'Lender\', \'Marine MP, LLC\', \'Arnold Schwarzenegger\', \'Fitness\', \'collectively, "MusclePh

In [35]:
pd.set_option('display.max_columns', None)

In [3]:
data.shape

(510, 83)

In [36]:
data.head()

,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,Expiration Date-Answer,Renewal Term,Renewal Term-Answer,Notice Period To Terminate Renewal,Notice Period To Terminate Renewal- Answer,Governing Law,Governing Law-Answer,Most Favored Nation,Most Favored Nation-Answer,Competitive Restriction Exception,Competitive Restriction Exception-Answer,Non-Compete,Non-Compete-Answer,Exclusivity,Exclusivity-Answer,No-Solicit Of Customers,No-Solicit Of Customers-Answer,No-Solicit Of Employees,No-Solicit Of Employees-Answer,Non-Disparagement,Non-Disparagement-Answer,Termination For Convenience,Termination For Convenience-Answer,Rofr/Rofo/Rofn,Rofr/Rofo/Rofn-Answer,Change Of Control,Change Of Control-Answer,Anti-Assignment,Anti-Assignment-Answer,Revenue/Profit Sharing,Revenue/Profit Sharing-Answer,Price Restrictions,Price Restrictions-Answer,Minimum Commitment,Minimum Commitment-Answer,Volume Restriction,Volume Restriction-Answer,Ip Ownership Assignment,Ip Ownership Assignment-Answer,Joint Ip Ownership,Joint Ip Ownership-Answer,License Grant,License Grant-Answer,Non-Transferable License,Non-Transferable License-Answer,Affiliate License-Licensor,Affiliate License-Licensor-Answer,Affiliate License-Licensee,Affiliate License-Licensee-Answer,Unlimited/All-You-Can-Eat-License,Unlimited/All-You-Can-Eat-License-Answer,Irrevocable Or Perpetual License,Irrevocable Or Perpetual License-Answer,Source Code Escrow,Source Code Escrow-Answer,Post-Termination Services,Post-Termination Services-Answer,Audit Rights,Audit Rights-Answer,Uncapped Liability,Uncapped Liability-Answer,Cap On Liability,Cap On Liability-Answer,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,12/31/14,['This agreement shall begin upon the date of ...,successive 1 year,['This Agreement may be terminated by either p...,30 days,['This Agreement is accepted by Company in the...,Nevada,[],No,[],No,[],No,[],No,[],No,[],No,['Company shall not specify the business pract...,Yes,[],No,[],No,[],No,"['MA may not assign, sell, lease or otherwise ...",Yes,[],No,[],No,['INITIAL ORDER COMMITMENT - MA commits to pur...,Yes,[],No,[],No,[],No,"['Company hereby grants MA, during the term of...",Yes,[],No,[],No,[],No,[],No,[],No,[],No,[],No,['MA shall keep accurate records of the sales ...,Yes,[],No,['The foregoing states the entire liability of...,Yes,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",6/30/10,"['At Rogers\' option, this Agreement shall ren...",2 years,"[""Notwithstanding the foregoing, if, at the ex...",60 days,"['This Agreement is subject to all laws, regul...","Ontario, Canada",['In the event that Licensor grants to another...,Yes,[],No,[],No,[],No,[],No,[],No,[],No,"[""Notwithstanding any other provision of this ...",Yes,[],No,[],No,"['This Agreement may not be assigned, sold or ...",Yes,['For so long as Rogers is required by Applica...,Yes,[],No,"['Licensor shall make available to Rogers, on ...",Yes,[],No,[],No,[],No,"['During the Term, Rogers shall have the non-e...",

In [74]:
data["Filename"].tolist()[0]

'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'

In [34]:
data.columns.tolist()

['Filename',
 'Document Name',
 'Document Name-Answer',
 'Parties',
 'Parties-Answer',
 'Agreement Date',
 'Agreement Date-Answer',
 'Effective Date',
 'Effective Date-Answer',
 'Expiration Date',
 'Expiration Date-Answer',
 'Renewal Term',
 'Renewal Term-Answer',
 'Notice Period To Terminate Renewal',
 'Notice Period To Terminate Renewal- Answer',
 'Governing Law',
 'Governing Law-Answer',
 'Most Favored Nation',
 'Most Favored Nation-Answer',
 'Competitive Restriction Exception',
 'Competitive Restriction Exception-Answer',
 'Non-Compete',
 'Non-Compete-Answer',
 'Exclusivity',
 'Exclusivity-Answer',
 'No-Solicit Of Customers',
 'No-Solicit Of Customers-Answer',
 'No-Solicit Of Employees',
 'No-Solicit Of Employees-Answer',
 'Non-Disparagement',
 'Non-Disparagement-Answer',
 'Termination For Convenience',
 'Termination For Convenience-Answer',
 'Rofr/Rofo/Rofn',
 'Rofr/Rofo/Rofn-Answer',
 'Change Of Control',
 'Change Of Control-Answer',
 'Anti-Assignment',
 'Anti-Assignment-Answer',

In [32]:
data["Price Restrictions-Answer"].tolist()

['No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',

In [23]:
import re
parties = re.findall(r'([^;()]+)(?:\s*\(|$)', data["Parties-Answer"][0])
parties = [party.strip() for party in parties if party.strip()] 
parties

['Birch First Global Investments Inc.', 'Mount Kowledge Holdings Inc.']

In [24]:
data["Notice Period To Terminate Renewal- Answer"][0]

'30 days'

In [39]:
data["Filename"][20]

'EcoScienceSolutionsInc_20171117_8-K_EX-10.1_10956472_EX-10.1_Endorsement Agreement.pdf'

In [50]:
len(data["Document Name-Answer"].unique().tolist())

285

In [ ]:
# from langchain_community.retrievers. import SelfQueryRetriever

ImportError: cannot import name 'SelfQueryRetriever' from 'langchain_core.retrievers' (c:\Users\elbou\Desktop\contractAssistant\contractAssistant\.venv\Lib\site-packages\langchain_core\retrievers.py)

In [71]:
len(list(data["Governing Law-Answer"].unique()))

78